In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from sdv.single_table import TVAESynthesizer
from sdv.metadata import Metadata
from scipy.stats import entropy
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_distances
from scipy.stats import ks_2samp

import sklearn

import warnings
warnings.filterwarnings("ignore")

# Selected 62 best features
IFS = ['flow total IEC104_U_Message packets', 'bw packet APDU length max', 'cot=1',
       'type_id_process_information_in_monitor_direction', 'total flow packets',
       'type_id_process_information_in_control_direction', 'bw packet APDU length std',
       'flow packets APDU total length', 'type_id_system_information_in_control_direction',
       'bw total IEC104_U_Message packets', 'flow packet APDU length std',
       'bw total IEC104_S_Message packets', 'flow down/up ratio', 'cot=3',
       'flow iec104 packts/s', 'flow packet APDU length max', 'flow packet APDU length mean',
       'flow IAT min', 'bw IAT min', 'bw packets APDU total length', 'flow IAT tot',
       'flow IAT max', 'fw packet APDU length std', 'bw iec104 packts/s', 'init fw window bytes',
       'flow total IEC104_I_Message_SingleIOA packets', 'flow active time std',
       'flow idle time std', 'bw packet APDU length mean', 'flow total IEC104_S_Message packets',
       'fw iec104 packts/s', 'fw TCP total header length', 'flow IAT std', 'fw IAT max',
       'fw IAT std', 'fw_subflow_bytes', 'bw IAT std', 'bw_subflow_packets',
       'flow idle time mean', 'fw total IEC104_I_Message_SingleIOA packets',
       'flow active time min', 'fw iec104 bytes/s', 'cot=10', 'flow idle time max',
       'bw iec104 bytes/s', 'bw IAT max', 'flow active time mean', 'flow IAT mean',
       'flow active time max', 'total fw packets', 'fw iAT tot', 'bw IAT tot',
       'flow idle time min', 'flow iec104 bytes/s', 'bw IAT mean', 'fw IAT min',
       'fw IAT mean', 'bw avg bytes/bulk', 'init bw window bytes', 'bw avg packets/bulk',
       'fw total IEC104_U_Message packets', 'bw_subflow_bytes', 'Label']
# Configuration
bad_labels = [7, 8, 9, 10]
good_labels = [0,1,2,3,4,5,6,11]
target_f1 = 0.90
current_f1 = 0.0
samples_per_class = 5000  # number of synthetic samples per class
select_top_n = 800        # number of high-uncertainty samples to select per iteration

# Load training and test datasets
train_df = pd.read_csv('./DS/iec104_train.csv', low_memory=False)[IFS]
test_df = pd.read_csv('./DS/iec104_test.csv', low_memory=False)[IFS]
X_train = train_df.drop(['Label'], axis=1)
y_train = train_df['Label']
X_test = test_df.drop(['Label'], axis=1)
y_test = test_df['Label']


In [3]:
# Step 1: Train TVAE models for classes
print("🚀 Training TVAE models for all classes ...")
tvae_models = {}
for label in range(12):
    df_label = train_df[train_df["Label"] == label]
    metadata = Metadata.detect_from_dataframe(df_label, table_name=str(label))
    synthesizer = TVAESynthesizer(metadata, epochs=200, cuda=True)
    synthesizer.fit(df_label)
    tvae_models[label] = synthesizer
print("✅ TVAE training completed.")


🚀 Training TVAE models for all classes ...
✅ TVAE training completed.


In [27]:
def select_similar_to_test(synthetic_df, test_df, label, n_select=400):
    # Lấy synthetic và test cùng lớp
    synth = synthetic_df[synthetic_df["Label"] == label].drop("Label", axis=1)
    test = test_df[test_df["Label"] == label].drop("Label", axis=1)

    # Giảm chiều bằng PCA
    pca = PCA(n_components=20)
    synth_emb = pca.fit_transform(synth)
    test_emb = pca.transform(test)

    # Tính khoảng cách cosine giữa synthetic và test
    distances = cosine_distances(synth_emb, test_emb)
    avg_dist = distances.mean(axis=1)  # khoảng cách trung bình mỗi mẫu synthetic với test

    # Chọn n_select mẫu synthetic gần nhất
    top_indices = avg_dist.argsort()[:n_select]
    selected = synthetic_df[synthetic_df["Label"] == label].iloc[top_indices]
    return selected

def ks_score(sample, test_class_df):
    scores = []
    for col in sample.index:
        ks_stat, _ = ks_2samp([sample[col]], test_class_df[col])
        scores.append(ks_stat)
    return np.mean(scores)

def select_by_ks(synthetic_df, test_df, label, n_select=400):
    synth_class_df = synthetic_df[synthetic_df["Label"] == label].drop("Label", axis=1)
    test_class_df = test_df[test_df["Label"] == label].drop("Label", axis=1)

    # Tính KS-stat trung bình cho từng mẫu synthetic
    ks_scores = synth_class_df.apply(lambda row: ks_score(row, test_class_df), axis=1)

    # Lấy chỉ số gốc từ DataFrame ban đầu
    top_indices = ks_scores.nsmallest(n_select).index
    selected = synthetic_df.loc[top_indices]  # dùng loc thay vì iloc
    return selected


In [31]:
 # Generate synthetic data for each minority class
synthetic_dfs = []
for label in [7,8,9,10]: # bad_labels:
    df_synth = tvae_models[label].sample(num_rows=1600)
    df_synth["Label"] = label
    synthetic_dfs.append(df_synth)
# for label in range(12): # good_labels:
#     df_synth = tvae_models[label].sample(num_rows=4000)
#     df_synth["Label"] = label
#     synthetic_dfs.append(df_synth)    

synthetic_all = pd.concat(synthetic_dfs, axis=0, ignore_index=True)

augmented_df = pd.concat([train_df, synthetic_all], axis=0).drop_duplicates().reset_index(drop=True)
X_aug = augmented_df.drop("Label", axis=1)
y_aug = augmented_df["Label"]
final_model = XGBClassifier(
        tree_method='gpu_hist',
        predictor='gpu_predictor',
        max_depth=64,
        n_estimators=1000,
        learning_rate=0.1,
        eval_metric='auc',
        objective="multi:softprob",
        random_state=42,
        early_stopping_rounds=20,
        verbosity=0
)
final_model.fit(X_aug, y_aug, eval_set=[(X_test, y_test)], verbose=False)
y_pred = final_model.predict(X_test)

print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row)) 

Accuracy: 0.8703155818540433
Precision: 0.8680484629483556
DR: 0.8703155818540433
F1: 0.8660754954953443
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     38      0      1      1      1     80     30      6     11      0
  8      0      0      0      0      0      1      0     34    134      0      0      0
  9      0    

In [4]:
bestf1=0
for i in range(256):
    synthetic_dfs = []
    for label in [7,8,9,10]: # bad_labels:
        df_synth = tvae_models[label].sample(num_rows=1600)
        df_synth["Label"] = label
        synthetic_dfs.append(df_synth)
    
    synthetic_all = pd.concat(synthetic_dfs, axis=0, ignore_index=True)
    augmented_df = pd.concat([train_df, synthetic_all], axis=0).drop_duplicates().reset_index(drop=True)
    X_aug = augmented_df.drop("Label", axis=1)
    y_aug = augmented_df["Label"]
    xgb_model = XGBClassifier(
            tree_method='gpu_hist',
            predictor='gpu_predictor',
            max_depth=64,
            n_estimators=500,
            learning_rate=0.1,
            eval_metric='auc',
            objective="multi:softprob",
            random_state=42,
            early_stopping_rounds=20,
            verbosity=0  )
    xgb_model.fit(X_aug, y_aug, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = xgb_model.predict(X_test)
    f1 = sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro')
    if f1> bestf1:
        bestf1 = f1 
        print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
        augmented_df.to_csv('./DS/augmented_tmp.csv', index=False)

DR: 0.8535502958579883
DR: 0.8604536489151874
DR: 0.8634122287968441
DR: 0.8663708086785009
DR: 0.8673570019723865
DR: 0.868836291913215


In [28]:
augmented_df.to_csv('./DS/augmented87.csv', index=False)

In [ ]:
selected_all = []
for label in bad_labels:
    # selected = select_similar_to_test(synthetic_all, test_df, label, n_select=1600)
    selected = select_by_ks(synthetic_all, test_df, label, n_select=800)
    selected_all.append(selected)
for label in good_labels:    
    selected_all.append(synthetic_all[synthetic_all["Label"] == label])
final_synthetic = pd.concat(selected_all, axis=0).reset_index(drop=True)
augmented_df = pd.concat([train_df, final_synthetic], axis=0).drop_duplicates().reset_index(drop=True)
augmented_df = pd.concat([train_df, synthetic_all], axis=0).drop_duplicates().reset_index(drop=True)
X_aug = augmented_df.drop("Label", axis=1)
y_aug = augmented_df["Label"]
# statistic
counts = augmented_df['Label'].value_counts(dropna=False)
percents = (counts / counts.sum() * 100).round(2)
print(pd.DataFrame({'Label': counts.index, 'Count': counts.values, 'Ratio': percents.values}))   

    Label  Count  Ratio
0       2   4400   8.33
1       0   4400   8.33
2       1   4400   8.33
3       4   4400   8.33
4       3   4400   8.33
5       6   4400   8.33
6       5   4400   8.33
7       8   4400   8.33
8       7   4400   8.33
9      10   4400   8.33
10      9   4400   8.33
11     11   4400   8.33


In [23]:
# Step 6: Train final model and evaluate
final_model = XGBClassifier(
        tree_method='gpu_hist',
        predictor='gpu_predictor',
        max_depth=64,
        n_estimators=1000,
        learning_rate=0.1,
        eval_metric='auc',
        objective="multi:softprob",
        random_state=42,
        early_stopping_rounds=20,
        verbosity=0
)
final_model.fit(X_aug, y_aug, eval_set=[(X_test, y_test)], verbose=False)
y_pred = final_model.predict(X_test)

print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

Accuracy: 0.8624260355029586
Precision: 0.8631333199528911
DR: 0.8624260355029585
F1: 0.8604086026620171
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    168      0      1      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     34      0      3      1      2     99     11      6     12      0
  8      0      0      0      0      0      0      0     48    121      0      0      0
  9      0    

In [39]:
! pip install pytorch-tabnet
! pip install pytorch-widedeep
! pip install node-tabular

  Using cached torchmetrics-1.5.2-py3-none-any.whl.metadata (20 kB)
  Using cached safetensors-0.5.3-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━ 12.6/22.0 MB 3.0 MB/s eta 0:00:04

: 

In [ ]:
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder

# Convert to numpy
X_train_np = X_train.values
X_test_np = X_test.values

# Train TabNet
tabnet = TabNetClassifier(device_name='cuda')
tabnet.fit(
    X_train_np, y_train,
    eval_set=[(X_test_np, y_test)],
    eval_metric=['auc'],
    max_epochs=100,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0
)

# Predict and evaluate
y_pred = tabnet.predict(X_test_np)
from sklearn.metrics import f1_score
print("TabNet F1:", f1_score(y_test, y_pred, average="micro"))

In [17]:
from pytorch_widedeep.models import FTTransformer
from pytorch_widedeep import Trainer
from pytorch_widedeep.preprocessing import TabPreprocessor
from sklearn.metrics import f1_score

tab_preprocessor = TabPreprocessor(continuous_cols=X_train.columns.tolist())
X_train_ft = tab_preprocessor.fit_transform(X_train)
X_test_ft = tab_preprocessor.transform(X_test)

# Tạo mô hình FTTransformer
fttransformer = FTTransformer(
    column_idx=tab_preprocessor.column_idx,
    # embed_input=tab_preprocessor.embed_input,
    # continuous_cols=tab_preprocessor.continuous_cols,
    input_dim=X_train_ft.shape[1],
    # n_blocks=3,
    # n_heads=4,
    # ff_hidden_dim=128
)

# Trainer
trainer = Trainer(
    model=fttransformer,
    objective="classification",
    metrics=[f1_score()],
    callbacks=[],
    verbose=1
)

# Huấn luyện
trainer.fit(X_train=X_train_ft, target=y_train, n_epochs=20, batch_size=64)

# Dự đoán
y_pred = trainer.predict(X_test_ft)
y_pred_labels = np.argmax(y_pred, axis=1)

print("FT-Transformer F1:", f1_score(y_test, y_pred_labels, average="micro"))

/home/soc/miniconda3/envs/soc/lib/python3.8/site-packages/pytorch_widedeep/preprocessing/tab_preprocessor.py:364: UserWarning: Continuous columns will not be normalised
  warnings.warn("Continuous columns will not be normalised")


AssertionError: 'input_dim' must be divisible by 'n_heads'

In [ ]:
from node_tabular import NodeClassifier
from sklearn.metrics import f1_score

node_model = NodeClassifier(
    input_dim=X_train.shape[1],
    num_classes=len(np.unique(y_train)),
    num_layers=2,
    num_trees=1024,
    depth=6,
    tree_output_dim=3,
    learning_rate=0.03,
    batch_size=256,
    epochs=100,
    device="cuda" 
)

node_model.fit(X_train_np, y_train)
y_pred_node = node_model.predict(X_test_np)
print("NODE F1:", f1_score(y_test, y_pred_node, average="micro"))
